In [1]:
import torch

In [2]:
x = torch.arange(4.0, requires_grad=True)
print(x)
print(x.grad)   # 梯度默认是None

tensor([0., 1., 2., 3.], requires_grad=True)
None


In [3]:
y = 2 * torch.dot(x, x)
y

tensor(28., grad_fn=<MulBackward0>)

In [4]:
# 现在y=2*x^2, 所以求y关于x的导数是4x
y.backward()  # 反向传播，计算
x.grad == 4 * x

tensor([True, True, True, True])

In [5]:
x.grad.zero_()  # 清除梯度
y = x.sum()
y.backward()  # 反向传播，计算
x.grad

tensor([1., 1., 1., 1.])

In [9]:
x.grad.zero_()  # 清除梯度
y = x ** 4
# 此时y不是标量，在调用backward时需要传入一个与y同形的参数，表示“雅可比向量积”
y.backward(torch.ones(len(x)))
x.grad

tensor([  0.,   4.,  32., 108.])

In [11]:
x.grad.zero_()  # 清除梯度
# 将某些计算结果从计算图中分离出来，不再具有梯度信息
y = x ** 3
u = y.detach()  # u不再具有梯度信息，被视作常数
z = u * x
z.sum().backward()
x.grad == u

tensor([True, True, True, True])

In [12]:
# 与此同时y仍然是计算图的一部分，因此它依然具有梯度信息
x.grad.zero_()
y.sum().backward()
x.grad == 3 * x ** 2

tensor([True, True, True, True])

In [17]:
def f(a):
    b = a * 3
    while b.norm() < 999:
        b = b * 4
    if b.sum() > 0:
        c = b
    else:
        c = 1100 * b
    return c

a = torch.randn(size=(), requires_grad=True)
d = f(a)
d.backward()  # 反向传播
a.grad == d / a     # d = C * a

tensor(True)